## Prepare questions from NIFD for training GRPO

Created by: Sahana Kowshik

Date: 05/07/2025

In [1]:
import pandas as pd
import json
import random

In [2]:
directory_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nifd"
train = pd.read_csv(f"{directory_path}/training_data/train.csv")

In [3]:
train.head()

,LONI_ID,CLINICAL_LINKDATE,DX,SITE,VISIT_NUMBER,DOB,GENDER,EDUCATION,RACE,CDR_DCDATE,...,RSMS_FTDADBEH,RSMS_FTDLYING,RSMS_FTDGOODF,RSMS_FTDREGUL,RSMS_FTDSMSCR,RSMS_FTDSPSCR,MDEXAM_DCDATE,MDEXAM_UHDRS,MDEXAM_UPDRS,patient_summary
0,1_S_0001,8/27/14,CON,UCSF,7,12/9/45,1,15,1,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{\n ""Gender"": ""Male"",\n ""Race"": ""White"",..."
1,1_S_0002,1/24/12,BV,UCSF,3,12/28/54,2,13,1,1/24/12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{\n ""Gender"": ""Female"",\n ""Race"": ""White..."
2,1_S_0003,8/20/12,SV,UCSF,4,6/8/44,1,12,1,8/15/12,...,NaN,NaN,NaN,NaN,NaN,NaN,8/15/12,12.0,14.0,"{\n ""Gender"": ""Male"",\n ""Years of Educat..."
3,1_S_0005,8/26/14,BV,UCSF,6,11/22/51,1,18,1,8/26/14,...,2.0,0.0,3.0,2.0,2.0,12.0,8/26/14,-5.0,15.0,"{\n ""Gender"": ""Male"",\n ""Race"": ""White"",..."
4,1_S_0006,7/15/11,BV,UCSF,4,7/24/48,1,15,99,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,7/15/11,-5.0,3.0,"{\n ""Gender"": ""Male"",\n ""Schwab and Engl..."


In [4]:
train['DX'].value_counts()

DX
CON                136
BV                  77
PNFA                40
SV                  39
PATIENT (OTHER)     39
L_SD                 4
Name: count, dtype: int64

In [5]:
columns = ['ID', 'patient_summary', 'diag_summary', 'question', 'options', 'ground_truth']

In [6]:
# Question variants for primary etiology identification
etiology_question_variants = [
    "Identify the primary etiology of the patient's cognitive impairment using the information provided.",
    "Based on the clinical details, determine the main cause of the patient's cognitive impairment.",
    "From the given patient information, select the primary underlying etiology for the cognitive symptoms.",
    "Using the data provided, identify the dominant cause of the patient's cognitive decline.",
    "Determine the primary etiology underlying the patient's cognitive impairment based on the provided information."
]

# Answer options for primary etiology prediction
etiology_options = """A. Alzheimer's disease (AD)
B. Lewy body disease (LBD)
C. Frontotemporal lobar degeneration and its variants, including primary progressive aphasia, corticobasal degeneration and progressive supranuclear palsy, and with or without amyotrophic lateral sclerosis (FTD)
D. Vascular brain injury or vascular dementia including stroke (VD)
E. Systemic and environmental factors including infectious diseases (HIV included), metabolic, substance abuse / alcohol, medications, systemic disease and delirium (SEF)
F. Psychiatric conditions including schizophrenia, depression, bipolar disorder, anxiety and posttraumatic stress disorder (PSY)
G. Traumatic brain injury (TBI)
H. Normal-pressure hydrocephalus (NPH)
I. Prion disease (CJD, other) 
J. Not applicable (no cognitive impairment)"""

In [7]:
# Function to assign ground truth label and MCI question
def add_etpr_question(row):
    if row['DX'] == 'CON':
        row['ground_truth'] = "J"
    elif row['DX'] in ['BV', 'PNFA', 'SV']:
        row['ground_truth'] = "C"

    # Randomly assign one of the question variants
    row['question'] = random.choice(etiology_question_variants)

    row['options'] = etiology_options
    return row

In [8]:
# Apply the function to the MCI training set
train = train.apply(add_etpr_question, axis=1)

In [9]:
train["ID"] = train["LONI_ID"]
train.drop(['LONI_ID'], axis=1, inplace=True)
train["diag_summary"] = train["DX"]
train.drop(['DX'], axis=1, inplace=True)
train = train[columns]
train = train.dropna(how='any').reset_index(drop=True)

/scratch/5552643.1.cds-gpu-long/ipykernel_3041042/3566189845.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train["ID"] = train["LONI_ID"]
/scratch/5552643.1.cds-gpu-long/ipykernel_3041042/3566189845.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train["diag_summary"] = train["DX"]


In [10]:
print(train.iloc[4]['patient_summary'])

{
    "Gender": "Male",
    "Schwab and England Activities of Daily Living - Score": 20,
    "Clinician's Global Impression of Change - Score": 6,
    "MD Exam - Unified Parkinson's Disease Rating Scale": 3
}


In [11]:
train['question'].value_counts()

question
From the given patient information, select the primary underlying etiology for the cognitive symptoms.             62
Using the data provided, identify the dominant cause of the patient's cognitive decline.                           60
Determine the primary etiology underlying the patient's cognitive impairment based on the provided information.    57
Based on the clinical details, determine the main cause of the patient's cognitive impairment.                     57
Identify the primary etiology of the patient's cognitive impairment using the information provided.                56
Name: count, dtype: int64

In [12]:
train['Q_TYPE'] = "Primary etiology"

In [13]:
train.to_csv(f"{directory_path}/training_data/training_data_grpo/train_with_questions.csv", index=False)

In [14]:
question_df = train.groupby(['question', 'Q_TYPE']).size().reset_index(name='count').sort_values(by=['Q_TYPE', 'count'])

In [15]:
# Save as CSV
csv_path = f"{directory_path}/training_data/training_data_grpo/train_question_counts.csv"
question_df.to_csv(csv_path, index=False)

In [16]:
len(train)

292